In [1]:
!pip install scikit-learn
!pip install numpy
!pip install torch
!pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ==============================
# 1. IMPORTS
# ==============================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

# Output folders
os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)

# ==============================
# 2. LOAD DATA
# ==============================

In [3]:
df = pd.read_csv("GEFCom2014_prepared.csv")

# Convert date if needed
df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["date", "hour"]
).reset_index(drop=True)

df.head()

,date,hour,load,temp,day_of_week,month,day,is_weekend,lag_1,lag_24,...,zone_industrial,sum_zones,temp_lag_1,temp_lag_24,temp_roll_mean_24,temp_roll_std_24,CDD,HDD,load_roll_mean_24,load_roll_std_24
0,2006-01-09,1,2800.0,20.666667,0,1,9,0,2970.0,2915.0,...,696.248834,2800.0,20.000000,15.000000,21.222222,4.400995,0.0,44.333333,3410.583333,482.767287
1,2006-01-09,2,2717.0,22.000000,0,1,9,0,2800.0,2803.0,...,654.756842,2717.0,20.666667,15.333333,21.458333,4.200083,0.0,43.000000,3405.791667,488.437436
2,2006-01-09,3,2690.0,23.333333,0,1,9,0,2717.0,2742.0,...,705.709645,2690.0,22.000000,16.000000,21.736111,3.992722,0.0,41.666667,3402.208333,493.342810
3,2006-01-09,4,2711.0,23.333333,0,1,9,0,2690.0,2730.0,...,692.137291,2711.0,23.333333,15.666667,22.041667,3.811136,0.0,41.666667,3400.041667,496.472643
4,2006-01-09,5,2814.0,24.333333,0,1,9,0,2711.0,2749.0,...,663.729587,2814.0,23.333333,16.333333,22.361111,3.567048,0.0,40.666667,3399.250000,497.601399


# ==============================
# 3. FEATURE GROUPS
# ==============================

In [4]:


target_col = "load"

# LOAD MODALITY (time memory)
load_features = [
    "load",
    "lag_1",
    "lag_24",
    "load_roll_mean_24",
    "load_roll_std_24"
]

# TEMPERATURE MODALITY (nonlinear effects)
temp_features = [
    "temp",
    "temp_lag_1",
    "temp_lag_24",
    "temp_roll_mean_24",
    "temp_roll_std_24",
    "CDD",
    "HDD"
]

# CALENDAR MODALITY (categorical)
calendar_features = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend"
]

# ==============================
# 4. FIX CATEGORICAL RANGES
# ==============================

In [5]:


df["hour"] = df["hour"].astype(int) % 24
df["day_of_week"] = df["day_of_week"].astype(int) % 7
df["month"] = df["month"].astype(int) - 1   # 0–11
df["is_weekend"] = df["is_weekend"].astype(int)

In [6]:
# ==============================
# 5. TRAIN-TEST SPLIT
# ==============================

split_idx = int(0.8 * len(df))

train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

# ==============================
# 6. NORMALIZATION
# ==============================

scaler = MinMaxScaler()

continuous_features = (
    load_features +
    temp_features
)

# Fit ONLY on training data
train_df[continuous_features] = scaler.fit_transform(
    train_df[continuous_features]
)

# Transform test data using TRAIN statistics
test_df[continuous_features] = scaler.transform(
    test_df[continuous_features]
)

# ==============================
# 7. SEQUENCE CREATION
# ==============================

SEQ_LEN = 24

def create_sequences(df):

    load_array = df[load_features].values.astype(np.float32)
    temp_array = df[temp_features].values.astype(np.float32)
    cal_array = df[calendar_features].values.astype(np.int64)
    target_array = df[target_col].values.astype(np.float32)

    n_samples = len(df) - SEQ_LEN

    # ---------------------------------
    # PREALLOCATE MEMORY
    # ---------------------------------
    X_load = np.zeros(
        (n_samples, SEQ_LEN, len(load_features)),
        dtype=np.float32
    )

    X_temp = np.zeros(
        (n_samples, SEQ_LEN, len(temp_features)),
        dtype=np.float32
    )

    X_cal = np.zeros(
        (n_samples, SEQ_LEN, len(calendar_features)),
        dtype=np.int64
    )

    y = np.zeros(
        n_samples,
        dtype=np.float32
    )

    # ---------------------------------
    # FILL ARRAYS
    # ---------------------------------
    for i in range(n_samples):

        X_load[i] = load_array[i:i+SEQ_LEN]

        X_temp[i] = temp_array[i:i+SEQ_LEN]

        X_cal[i] = cal_array[i:i+SEQ_LEN]

        y[i] = target_array[i+SEQ_LEN]

    return X_load, X_temp, X_cal, y

# Create sequences separately
Xl_train, Xt_train, Xc_train, y_train = create_sequences(train_df)

Xl_test, Xt_test, Xc_test, y_test = create_sequences(test_df)

# ==============================
# 8. DATASET CLASS
# ==============================

class LoadDataset(Dataset):

    def __init__(self, Xl, Xt, Xc, y):

        self.Xl = torch.tensor(
            Xl,
            dtype=torch.float32
        )

        self.Xt = torch.tensor(
            Xt,
            dtype=torch.float32
        )

        self.Xc = torch.tensor(
            Xc,
            dtype=torch.long
        )

        self.y = torch.tensor(
            y,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.y)

    def __getitem__(self, idx):

        return (

            self.Xl[idx],
            self.Xt[idx],
            self.Xc[idx],
            self.y[idx]

        )

# ==============================
# 9. DATALOADERS
# ==============================

train_loader = DataLoader(

    LoadDataset(
        Xl_train,
        Xt_train,
        Xc_train,
        y_train
    ),

    batch_size=64,
    shuffle=False

)

test_loader = DataLoader(

    LoadDataset(
        Xl_test,
        Xt_test,
        Xc_test,
        y_test
    ),

    batch_size=64,
    shuffle=False

)

# ==============================
# 9. Fusion MODEL
# ==============================

In [7]:


class MultimodalModel(nn.Module):
    def __init__(self):
        super().__init__()

        # LOAD ENCODER (LSTM)
        self.lstm = nn.LSTM(
            input_size=len(load_features),
            hidden_size=64,
            batch_first=True
        )

        # TEMP ENCODER (MLP)
        self.temp_mlp = nn.Sequential(
            nn.Linear(len(temp_features), 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        # CALENDAR EMBEDDINGS
        self.hour_emb = nn.Embedding(24, 8)
        self.day_emb = nn.Embedding(7, 4)
        self.month_emb = nn.Embedding(12, 4)
        self.weekend_emb = nn.Embedding(2, 2)

        fusion_size = 64 + 32 + (8 + 4 + 4 + 2)

        self.fc = nn.Sequential(
            nn.Linear(fusion_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, Xl, Xt, Xc):

        # Load → LSTM
        _, (h, _) = self.lstm(Xl)
        load_out = h[-1]

        # Temp → MLP (last timestep)
        temp_out = self.temp_mlp(Xt[:, -1, :])

        # Calendar embeddings
        hour = self.hour_emb(Xc[:, -1, 0])
        day = self.day_emb(Xc[:, -1, 1])
        month = self.month_emb(Xc[:, -1, 2])
        weekend = self.weekend_emb(Xc[:, -1, 3])

        cal_out = torch.cat([hour, day, month, weekend], dim=1)

        # Fusion
        fused = torch.cat([load_out, temp_out, cal_out], dim=1)

        return self.fc(fused)

# ==============================
# 9. Uni-MODEL
# ==============================

In [8]:
class LoadOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size=len(load_features),
            hidden_size=64,
            batch_first=True
        )
        
        self.fc = nn.Linear(64, 1)

    def forward(self, Xl):
        _, (h, _) = self.lstm(Xl)
        return self.fc(h[-1])

class TempOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(len(temp_features), 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, Xt):
        return self.net(Xt[:, -1, :])
    

class CalendarOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.hour_emb = nn.Embedding(24, 8)
        self.day_emb = nn.Embedding(7, 4)
        self.month_emb = nn.Embedding(12, 4)
        self.weekend_emb = nn.Embedding(2, 2)

        self.fc = nn.Sequential(
            nn.Linear(8+4+4+2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, Xc):
        hour = self.hour_emb(Xc[:, -1, 0])
        day = self.day_emb(Xc[:, -1, 1])
        month = self.month_emb(Xc[:, -1, 2])
        weekend = self.weekend_emb(Xc[:, -1, 3])

        x = torch.cat([hour, day, month, weekend], dim=1)
        return self.fc(x)
    



In [9]:
def train_model(model, train_loader, mode, epochs=10):
    model.train()

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        epoch_loss = 0

        for Xl, Xt, Xc, yb in train_loader:
            optimizer.zero_grad()

            if mode == "load":
                preds = model(Xl).squeeze()

            elif mode == "temp":
                preds = model(Xt).squeeze()

            elif mode == "cal":
                preds = model(Xc).squeeze()

            elif mode == "fusion":
                preds = model(Xl, Xt, Xc).squeeze()

            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"{mode.upper()} Epoch {epoch+1}: {epoch_loss:.4f}")

    return model

In [10]:
def evaluate_model(model, loader, mode):
    model.eval()
    preds, actuals = [], []

    with torch.no_grad():
        for Xl, Xt, Xc, yb in loader:

            if mode == "load":
                out = model(Xl)

            elif mode == "temp":
                out = model(Xt)

            elif mode == "cal":
                out = model(Xc)

            elif mode == "fusion":
                out = model(Xl, Xt, Xc)

            preds.extend(out.squeeze().numpy())
            actuals.extend(yb.numpy())

    mae = mean_absolute_error(actuals, preds)
    rmse = np.sqrt(mean_squared_error(actuals, preds))

    return mae, rmse, preds, actuals

# ==============================
# 10. TRAINING
# ==============================

In [11]:
# ==============================
# 10. TRAINING
# ==============================

load_model = LoadOnlyModel()
load_model = train_model(load_model, train_loader, "load")

temp_model = TempOnlyModel()
temp_model = train_model(temp_model, train_loader, "temp")

cal_model = CalendarOnlyModel()
cal_model = train_model(cal_model, train_loader, "cal")

fusion_model = MultimodalModel()
fusion_model = train_model(fusion_model, train_loader, "fusion")

LOAD Epoch 1: 5.5565
LOAD Epoch 2: 0.7367
LOAD Epoch 3: 0.4437
LOAD Epoch 4: 0.2912
LOAD Epoch 5: 0.2350
LOAD Epoch 6: 0.2096
LOAD Epoch 7: 0.1902
LOAD Epoch 8: 0.1708
LOAD Epoch 9: 0.1527
LOAD Epoch 10: 0.1465
TEMP Epoch 1: 22.9517
TEMP Epoch 2: 16.1560
TEMP Epoch 3: 15.8623
TEMP Epoch 4: 15.7925
TEMP Epoch 5: 15.7159
TEMP Epoch 6: 15.6657
TEMP Epoch 7: 15.5845
TEMP Epoch 8: 15.4947
TEMP Epoch 9: 15.3707
TEMP Epoch 10: 15.1838
CAL Epoch 1: 14.5145
CAL Epoch 2: 6.3254
CAL Epoch 3: 5.0573
CAL Epoch 4: 4.7075
CAL Epoch 5: 4.5604
CAL Epoch 6: 4.4723
CAL Epoch 7: 4.3989
CAL Epoch 8: 4.3254
CAL Epoch 9: 4.2374
CAL Epoch 10: 4.1512
FUSION Epoch 1: 6.7563
FUSION Epoch 2: 1.5549
FUSION Epoch 3: 0.9771
FUSION Epoch 4: 0.6272
FUSION Epoch 5: 1.1926
FUSION Epoch 6: 0.5902
FUSION Epoch 7: 0.7466
FUSION Epoch 8: 0.5543
FUSION Epoch 9: 0.4220
FUSION Epoch 10: 0.5688


# ==============================
# 11. EVALUATION
# ==============================

In [12]:
# ==============================
# 11. FINAL EVALUATION
# ==============================

results = []

# LOAD MODEL
mae_load, rmse_load, load_preds, actuals = evaluate_model(load_model, test_loader, "load")
results.append(["Load Only", mae_load, rmse_load])

# TEMP MODEL
mae_temp, rmse_temp, temp_preds, _ = evaluate_model(temp_model, test_loader, "temp")
results.append(["Temp Only", mae_temp, rmse_temp])

# CALENDAR MODEL
mae_cal, rmse_cal, cal_preds, _ = evaluate_model(cal_model, test_loader, "cal")
results.append(["Calendar Only", mae_cal, rmse_cal])

# FUSION MODEL
mae_fusion, rmse_fusion, fusion_preds, _ = evaluate_model(fusion_model, test_loader, "fusion")
results.append(["Fusion Model", mae_fusion, rmse_fusion])

# ==============================
# 12. SAVE OUTPUTS
# ==============================

In [13]:
# ==============================
# 12. COMPARISON TABLE
# ==============================

comparison_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE"])

print(comparison_df)

# Save
comparison_df.to_csv("outputs/tables/rq1_model_comparison.csv", index=False)

           Model       MAE      RMSE
0      Load Only  0.012514  0.016399
1      Temp Only  0.131118  0.158296
2  Calendar Only  0.050936  0.065570
3   Fusion Model  0.017604  0.023811


# ==============================
# 13. SAVE FIGURE
# ==============================

In [14]:
# ==============================
# 13. MODEL COMPARISON PLOT
# ==============================

plt.figure()

plt.bar(comparison_df["Model"], comparison_df["RMSE"])
plt.ylabel("RMSE")
plt.title("RQ1: Model Comparison")

plt.xticks(rotation=30)

plt.savefig("outputs/figures/rq1_model_comparison.pdf")
plt.close()

In [15]:
# ==============================
# 14. OVERLAY PLOT
# ==============================

plt.figure()

plt.plot(actuals[:200], label="Actual", linewidth=2)
plt.plot(fusion_preds[:200], label="Fusion", linewidth=2)

plt.plot(load_preds[:200], label="Load Only", linestyle="--")
plt.plot(temp_preds[:200], label="Temp Only", linestyle="--")
plt.plot(cal_preds[:200], label="Calendar Only", linestyle="--")

plt.legend()
plt.title("Fusion vs Unimodal Models")

plt.savefig("outputs/figures/rq1_overlay_comparison.pdf")
plt.close()

In [16]:
# ==============================
# 15. SAVE PREDICTIONS
# ==============================

predictions_df = pd.DataFrame({
    "Actual": actuals,
    "Fusion": fusion_preds,
    "Load Only": load_preds,
    "Temp Only": temp_preds,
    "Calendar Only": cal_preds
})

predictions_df.to_csv("outputs/tables/rq1_all_predictions.csv", index=False)

In [17]:

fusion_errors = np.array(actuals) - np.array(fusion_preds)
load_errors = np.array(actuals) - np.array(load_preds)

plt.figure()

plt.hist(fusion_errors, bins=50, alpha=0.6, label="Fusion")
plt.hist(load_errors, bins=50, alpha=0.6, label="Load Only")

plt.legend()
plt.title("Error Distribution Comparison")

plt.savefig("outputs/figures/rq1_error_distribution.pdf")
plt.close()